# Mandatory Disclosure and Housing Market Dynamics
## Synthetic DGP Exploration — Patras (2024)

This notebook explores the structural model of mandatory energy-performance-certificate (EPC) disclosure in housing markets. Buyers are heterogeneous in their EPC sensitivity $\theta \sim U[0,1]$; a fraction $\pi$ are *informed* (know their own $\theta$) while the rest are *uninformed*. Sellers decide whether to hide their EPC rating at a cost $c_0$. In equilibrium, a hiding threshold $\sigma_0^*$ separates sellers who disclose from those who hide, while buyers hold rational beliefs about the distribution of hidden ratings. The two key structural parameters are:

| Parameter | Symbol | Interpretation |
|-----------|--------|----------------|
| Buyer EPC sensitivity share | $\rho$ | Share of buyers whose willingness-to-pay is strictly increasing in the EPC rating |
| Hiding cost | $c_0$ | Per-unit cost (fraction of property value) paid by a seller to conceal the EPC rating |

In [ ]:
import logging
import warnings
import numpy as np
import pandas as pd

warnings.filterwarnings("ignore")
logging.basicConfig(level=logging.INFO, format="%(levelname)s: %(message)s")

from src.viz import (
    chart_data_overview, chart_hedonic_params, chart_hiding_premiums,
    chart_moment_fit, chart_estimates_by_market, chart_equilibrium_objects,
    chart_msm_surface, chart_belief_function, chart_renovation_thresholds,
    styled_df,
)

from src.data_prep  import make_synthetic_data, compute_market_stats, compute_market_subsidies
from src.hedonic    import (estimate_hedonic, calibrate_demand_scale,
                             calibrate_demand_scale_m, compute_hiding_premium,
                             estimate_rho_direct)
from src.moments    import extract_data_moments, build_weighting_matrix
from src.estimation import (build_valuation_params, estimate_msm, msm_objective,
                             compute_market_subsidies)
from src.model      import solve_equilibrium, belief_function
from src.validation import run_all_validation

## Configuration

The seven key DGP and estimation parameters are:

| Parameter | Description |
|-----------|-------------|
| `N_LISTINGS` | Total number of simulated property listings |
| `N_MARKETS` | Number of local housing markets (e.g., urban zones) |
| `PI` | Share of *informed* buyers ($\pi$) |
| `THETA_BAR` | Quality cutoff: sellers with $\theta_i = 1$ are high-quality, $\theta_i = 0$ are low-quality ($\bar{\theta}$) |
| `RHO_TRUE` | True share of EPC-sensitive buyers ($\rho$) used to generate data |
| `BETA_1_TRUE` | True hedonic EPC coefficient ($\beta_1$) used to generate prices |
| `TARGET_SIGMA_0` | Target equilibrium hiding threshold $\sigma_0^*$ — controls hiding prevalence in DGP |

Additional settings control the optimization (`WEIGHTING`, `C0_INIT`, `N_RESTARTS`) and the hedonic control variables (`CONTROLS`).

In [ ]:
# --- DGP parameters ---
N_LISTINGS     = 100_000
N_MARKETS      = 20
PI             = 0.60        # share of informed buyers
THETA_BAR      = 0.40        # mandatory-disclosure threshold
RHO_TRUE       = 0.80        # true buyer EPC-sensitivity share
BETA_1_TRUE    = 0.20        # true hedonic EPC coefficient
TARGET_SIGMA_0 = 0.70        # target equilibrium hiding threshold
SEED           = 42

# --- Hedonic controls ---
CONTROLS = ["size_m2", "built_year", "cateaav2020_label", "distance"]

# --- Estimation settings ---
WEIGHTING  = "optimal_gmm"
C0_INIT    = 0.005
N_RESTARTS = 5

## Section 1 — Structural Data-Generating Process

The synthetic data is generated in five steps:

1. **Draw buyer types.** For each listing $i$ in market $m$, draw $K_i \sim \text{Bernoulli}(\pi)$ (informed indicator) and $\theta_i \sim U[0,1]$ (EPC sensitivity).
2. **High-quality sellers always disclose.** Sellers with $\theta_i = 1$ have no incentive to hide — disclosure is strictly profitable for them. In equilibrium $\sigma_1^* = 0$ (Proposition 3.1): all informed high-quality sellers reveal their certificate.
3. **Low-quality informed sellers face a hiding decision.** For $\theta_i \leq \bar{\theta}$, a seller hides iff the hiding gain exceeds cost. In equilibrium the hiding threshold $\sigma_0^*$ solves:
$$\Delta(e^*;\, \rho) = c_0$$
where $\Delta(e;\rho) = \rho \bigl[ v_1^{\text{hid}}(e) - v_1^{\text{dis}}(\sigma_0^*) \bigr]$ is the price premium from hiding and $e^*$ is the equilibrium buyer belief about hidden EPC ratings.
4. **Equilibrium beliefs.** Uninformed buyers form rational (Bayesian) beliefs $e^*(\sigma_0^*)$ about the distribution of hidden ratings.
5. **Price generation.** Log prices are drawn around the equilibrium price $p^*$ plus idiosyncratic noise $\varepsilon_i \sim N(0, \sigma_\varepsilon^2)$.

In [ ]:
# Generate synthetic data from the structural model
df, market_stats = make_synthetic_data(
    n_listings=N_LISTINGS,
    n_markets=N_MARKETS,
    pi=PI,
    theta_bar=THETA_BAR,
    rho_true=RHO_TRUE,
    beta_1_true=BETA_1_TRUE,
    target_sigma_0=TARGET_SIGMA_0,
    seed=SEED,
    controls=CONTROLS,
)

# Restrict to markets with sufficient variation for identification
estimable_markets = market_stats.index[market_stats["estimable"] == True].tolist()

print(f"Total listings         : {len(df):,}")
print(f"Markets generated      : {N_MARKETS}")
print(f"Estimable markets      : {len(estimable_markets)}")
print(f"Disclosure rate        : {df['d'].mean():.1%}")
print(f"Hiding rate (<=theta_bar) : {(df.loc[df['theta'] <= THETA_BAR, 'd'] == 0).mean():.1%}")

In [ ]:
# Listing group counts
group_counts = df.groupby("group").size().rename("count").to_frame()
group_counts["share"] = group_counts["count"] / len(df)
print(group_counts.to_string())

# Market-level summary statistics
market_stats.loc[estimable_markets].describe().T.round(4)

In [ ]:
fig = chart_data_overview(df, market_stats)
fig.show()

## Section 2 — Hedonic Regression

### Equation (110)
$$\log p_i = \alpha_m + \beta_1 \theta_i + X_i' \gamma + \varepsilon_i$$

The hedonic regression is estimated **only on truthful disclosers** (sellers who voluntarily reveal their EPC rating). Restricting to this sub-sample avoids selection bias from hiding: sellers who hide have systematically lower EPC ratings and their observed prices would conflate the hiding premium with the hedonic gradient.

**Attenuation bias.** Even on truthful disclosers, OLS gives a downward-biased estimate $\hat{\beta}_1 < \beta_1^{\text{true}}$ because:
- A fraction $(1-\pi)$ of buyers are uninformed and do not adjust their bid for $\theta_i$, flattening the price-quality gradient in equilibrium.
- The effective hedonic slope is approximately $\beta_1 \cdot \rho \cdot \pi$, so the attenuation factor is $(1 - \rho\pi)$.

**Demand scale calibration.** The raw $\hat{\beta}_1$ must be rescaled to recover the true willingness-to-pay gradient used in the structural model. Both a pooled scale and market-specific scales are computed.

In [ ]:
# Estimate hedonic model on truthful disclosers only
hedonic_params = estimate_hedonic(df, market_col="market_id", controls=CONTROLS)

# Calibrate demand scales (pooled and market-specific)
scale   = calibrate_demand_scale(hedonic_params)
scale_m = calibrate_demand_scale_m(hedonic_params)

attenuation = (BETA_1_TRUE - hedonic_params.beta_1) / BETA_1_TRUE

print(f"True beta_1              : {BETA_1_TRUE:.4f}")
print(f"Estimated beta_hat_1     : {hedonic_params.beta_1:.4f}")
print(f"Attenuation bias         : {attenuation:.1%}")
print(f"Pooled demand scale      : {scale:.4f}")
print(f"Hedonic R-squared        : {hedonic_params.fitted_model.rsquared:.4f}")
print(f"sigma_eps                : {hedonic_params.sigma_eps:.4f}")
print(f"N observations           : {hedonic_params.n_obs:,}")

In [ ]:
fig = chart_hedonic_params(
    hedonic_params,
    market_stats.loc[estimable_markets],
    true_beta1=BETA_1_TRUE,
)
fig.show()

### Direct $\rho$ Estimator

Before the full MSM, a market-level *hiding premium* $\hat{\Delta}_m$ is computed — the average price gap between hidden and disclosed listings after controlling for observable quality. This premium is proportional to $\rho$ in the model, yielding a closed-form direct estimator $\hat{\rho}^{\text{dir}}_m$ that serves as the MSM starting point.

In [ ]:
# Compute hiding premium and direct rho estimator
df_hiders, market_premium = compute_hiding_premium(df, hedonic_params)
rho_direct_df = estimate_rho_direct(market_stats.loc[estimable_markets], market_premium, hedonic_params)

print(f"Mean hiding premium delta_bar_m : {rho_direct_df['delta_bar'].mean():.4f}")
print(f"Mean rho_hat (direct)           : {rho_direct_df['rho_hat_clipped'].mean():.4f}  (true: {RHO_TRUE})")

fig = chart_hiding_premiums(rho_direct_df)
fig.show()

## Section 3 — MSM Estimation

### Moments

Five moments are matched per market $m$:

| Moment | Formula | Primary identification |
|--------|---------|------------------------|
| $m_1$ | Avg. log price of *disclosed* listings | Anchors $\alpha_m$, $\beta_1$ |
| $m_2$ | Avg. log price of *hidden* listings | Identifies hiding premium $\to$ $\rho$ |
| $m_3$ | Share of listings that are hidden | Identifies $\sigma_0^*$ $\to$ $c_0$ |
| $m_4$ | Variance of log prices among disclosers | Identifies $\sigma_\varepsilon$, $\beta_1$ |
| $m_5$ | Covariance of $\theta$ and log price (disclosers) | Over-identifying restriction on $\beta_1$ |

### MSM Objective

$$Q(\rho, c_0) = \sum_m \left( \hat{\mathbf{m}}_m - \mathbf{m}_m^{\text{model}}(\rho_m, c_{0m}) \right)' W_m \left( \hat{\mathbf{m}}_m - \mathbf{m}_m^{\text{model}}(\rho_m, c_{0m}) \right)$$

where $W_m$ is the optimal GMM weighting matrix (inverse of the moment sampling covariance). Moment $m_5$ is over-identifying: with only $\rho_m$ and $c_{0m}$ to estimate per market, five moments over-determine the system. The $J$-statistic $n \cdot Q^*$ is asymptotically $\chi^2(3)$ per market under correct specification.

In [ ]:
# Extract empirical moments per market
data_moments = extract_data_moments(df, market_col="market_id")
data_moments = {k: v for k, v in data_moments.items() if k in estimable_markets}

# Build structural valuation parameters per market
market_params = {}
for mid in estimable_markets:
    vp = build_valuation_params(
        market_id=mid,
        market_stats=market_stats,
        hedonic_params=hedonic_params,
        scale=scale,
        scale_m=scale_m,
    )
    market_params[mid] = vp

# Initialise rho from the direct estimator (warm start)
rho_init = rho_direct_df["rho_hat_clipped"].to_dict()
rho_init = {m: float(rho_init.get(m, 0.5)) for m in data_moments}

print(f"Markets entering MSM   : {len(estimable_markets)}")
print(f"Mean rho_init (direct) : {np.mean(list(rho_init.values())):.4f}")

In [ ]:
# Run MSM estimation
msm_result = estimate_msm(
    data_moments=data_moments,
    market_params=market_params,
    market_stats=market_stats.loc[estimable_markets],
    weighting_scheme=WEIGHTING,
    df=df,
    rho_init=rho_init,
    c0_init=C0_INIT,
    n_restarts=N_RESTARTS,
    seed=SEED,
)

rho_vals = list(msm_result.rho_hat.values())
c0_vals  = list(msm_result.c0_hat.values())

print(f"Converged              : {msm_result.converged}")
print(f"Q* (objective value)   : {msm_result.Q_star:.6f}")
print(f"Mean rho_hat           : {np.mean(rho_vals):.4f}  (true: {RHO_TRUE})")
print(f"Mean c0_hat            : {np.mean(c0_vals):.4f}")
print(f"Std  rho_hat           : {np.std(rho_vals):.4f}")
print(f"Std  c0_hat            : {np.std(c0_vals):.4f}")

In [ ]:
from src.moments import moment_summary_table
moment_summary_table(data_moments, msm_result.model_moments).round(4)

## Section 4 — Results

A good fit means model-implied moments track data moments closely across all markets. Systematic deviations in $m_2$ (hidden prices) or $m_3$ (hiding rate) point to misspecification in the equilibrium or the hiding cost distribution. The cross-market chart checks whether $\hat{\rho}_m$ and $\hat{c}_{0m}$ are stable across markets — as expected from a pooled DGP with homogeneous parameters — and whether the equilibrium type (pooling vs. separating) is consistent with the data patterns.

In [ ]:
fig = chart_moment_fit(data_moments, msm_result.model_moments)
fig.show()

In [ ]:
fig = chart_estimates_by_market(msm_result)
fig.show()

eq_counts = pd.Series(msm_result.eq_types).value_counts()
print("\nEquilibrium regime breakdown:")
print(eq_counts.to_string())

## Section 5 — Equilibrium Explorer

### How $\sigma_0^*(\rho)$ varies

The equilibrium hiding threshold $\sigma_0^*$ solves $\Delta(e^*(\sigma_0);\, \rho) = c_0$. As $\rho$ rises (more EPC-sensitive buyers), the hiding premium $\Delta$ rises, making it worthwhile for sellers with *lower* EPC ratings to hide — so $\sigma_0^*$ increases. Conversely, higher $c_0$ compresses the hiding region. The belief function $e^*(\sigma_0) = E[\theta \mid \theta \leq \sigma_0]$ is increasing in $\sigma_0$, creating strategic complementarities.

### What the MSM surface reveals about identification

The objective surface $Q(\rho, c_0)$ over a grid reveals:
- **Ridge vs. valley geometry.** A narrow valley indicates strong joint identification; a flat ridge suggests that the two parameters are only weakly identified jointly.
- **Global minimum location.** With simulated data the minimum should be near `(RHO_TRUE, c0_true)` if the model is correctly specified.
- **Multiple local minima.** Can arise from equilibrium multiplicity; `N_RESTARTS` guards against convergence to a local minimum during estimation.

In [ ]:
# Select the first estimable market for the explorer
m_explore = sorted(data_moments.keys())[0]

pi_m    = float(market_stats.loc[m_explore, "pi_hat"])
theta_m = float(market_stats.loc[m_explore, "theta_bar_hat"])

print(f"Exploring market   : {m_explore}")
print(f"  pi_m             : {pi_m:.3f}")
print(f"  theta_bar_m      : {theta_m:.3f}")
print(f"  rho_hat (MSM)    : {msm_result.rho_hat.get(m_explore):.4f}")
print(f"  c0_hat  (MSM)    : {msm_result.c0_hat.get(m_explore):.4f}")

fig = chart_equilibrium_objects(
    market_id=m_explore,
    pi=pi_m,
    theta_bar=theta_m,
    params=market_params[m_explore],
    c0_hat=msm_result.c0_hat.get(m_explore),
    rho_hat=msm_result.rho_hat.get(m_explore),
)
fig.show()

In [ ]:
# Compute MSM objective surface for the selected market (40x40 grid)
N_GRID   = 40
rho_grid = np.linspace(0.40, 0.99, N_GRID)
c0_grid  = np.linspace(0.001, 0.05, N_GRID)

vp_m   = market_params[m_explore]
df_sub = df[df["market_id"] == m_explore].copy()

wt_m = build_weighting_matrix(
    {m_explore: data_moments[m_explore]},
    df_sub,
    WEIGHTING,
)

Q_surface = np.full((N_GRID, N_GRID), np.nan)
for i, rho_v in enumerate(rho_grid):
    for j, c0_v in enumerate(c0_grid):
        Q_surface[i, j] = msm_objective(
            np.array([rho_v, c0_v]),
            {m_explore: data_moments[m_explore]},
            {m_explore: vp_m},
            market_stats.loc[estimable_markets],
            wt_m,
        )

fig = chart_msm_surface(
    rho_grid=rho_grid,
    c0_grid=c0_grid,
    Q_surface=Q_surface,
    rho_hat=msm_result.rho_hat.get(m_explore),
    c0_hat=msm_result.c0_hat.get(m_explore),
    market_id=m_explore,
)
fig.show()

In [ ]:
# Belief function chart for the selected market
sigma_0_star_m = float(market_stats.loc[m_explore, "sigma_0_hat"])

pi_vals = [0.3, 0.5, pi_m, 0.8]

fig = chart_belief_function(
    pi_vals=pi_vals,
    theta_bar=theta_m,
    sigma_0_star=sigma_0_star_m,
)
fig.show()

## Section 6 — Counterfactual Subsidies

### Policy-Relevant Renovation Cost (Eq. 130)

The model identifies the marginal hiding cost $c_0$ per unit of property value. A natural counterfactual asks: what subsidy $S^*_m$ would induce all sellers below $\bar{\theta}$ to renovate (and thus disclose)?

The break-even renovation cost under the model is:
$$\bar{R}_m = \hat{\rho}_m \cdot \hat{\beta}_1 \cdot \bar{p}_m \cdot \left(1 - \hat{\sigma}_{0m}^* \right)$$

where $\bar{p}_m$ is the average price in market $m$. The required public subsidy is then:
$$S^*_m = \max\left\{ \hat{R}_m - \bar{R}_m,\ 0 \right\}$$

Markets with high hiding rates and low EPC sensitivity require the largest subsidies to shift the equilibrium toward full disclosure.

In [ ]:
subsidy_df = compute_market_subsidies(
    market_stats=market_stats.loc[estimable_markets],
    market_params=market_params,
    rho_hat=msm_result.rho_hat,
    c0_hat=msm_result.c0_hat,
)

subsidy_df.describe().T.round(4)

fig = chart_renovation_thresholds(subsidy_df)
fig.show()

## Section 7 — Validation

Three validation checks assess model fit and internal consistency:

1. **Moment parity.** Are model-implied moments within two standard errors of their data counterparts in every market?
2. **Equilibrium consistency.** Does the recovered $\sigma_0^*$ satisfy the equilibrium fixed-point equation $\Delta(e^*(\sigma_0^*); \hat{\rho}_m) = \hat{c}_{0m}$?
3. **Hedonic back-out.** Does plugging estimated parameters back into the hedonic equation recover the observed price distribution (mean and variance)?

In [ ]:
validation = run_all_validation(
    df=df,
    hedonic_params=hedonic_params,
    market_stats=market_stats.loc[estimable_markets],
    msm_result=msm_result,
)
print(validation)